# Part 3 — 모델·토크나이저 선정

**목적** seq2seq 사전학습 모델을 고르고, 그 선택의 근거를 숫자로 남긴다.

**실행한 것** `python scripts/check_tokenizer.py` → `artifacts/tokenizer_report.json`
후보 4개의 토크나이저로 dev 1,500건(전처리 `speaker/drop`)을 인코딩해 비교했다. test(공식 valid)는 쓰지 않았다.

**무엇을 보나**
- **UNK 비율**: 토크나이저가 모르는 글자는 모델이 읽지도, 쓰지도 못한다
- **정답 되살림**: `decode(encode(정답)) == 정답` 비율. 되살리지 못하는 표현은 모델이 생성할 수 없다
- **토큰 길이**: 짧을수록 빠르고 메모리를 덜 쓴다

In [1]:
import json, unicodedata
from pathlib import Path
import pandas as pd
rep = json.loads(Path("../artifacts/tokenizer_report.json").read_text(encoding="utf-8"))
rows = {}
for r in rep["tokenizers"]:
    rows[r["model"]] = {"vocab": r["vocab_size"], "d_model": r["d_model"], "층": r["num_layers"],
        "source 평균": r["source_tokens"]["mean"], "source p99": r["source_tokens"]["p99"],
        "글자/토큰": r["source_chars_per_token"], "UNK 비율 %": r["source_unk_rate_pct"],
        "UNK 포함 대화 %": r["source_texts_with_unk_pct"], "정답 UNK 토큰": r["target_unk_tokens"],
        "정답 되살림 %": r["target_roundtrip_pct"]}
pd.DataFrame(rows).T

,vocab,d_model,층,source 평균,source p99,글자/토큰,UNK 비율 %,UNK 포함 대화 %,정답 UNK 토큰,정답 되살림 %
paust/pko-t5-base,50358.0,768.0,12.0,147.9,319.02,1.235,0.0000,0.00,0.0,99.73
paust/pko-t5-small,50358.0,512.0,8.0,147.9,319.02,1.235,0.0000,0.00,0.0,99.73
KETI-AIR/ke-t5-base,64100.0,768.0,12.0,100.7,210.01,1.814,1.0163,56.20,27.0,97.53
google/mt5-base,250100.0,768.0,12.0,129.3,280.05,1.413,0.3414,30.67,2.0,99.13


## 1. ke-t5 — 가장 짧지만 구어체를 모른다

입력이 가장 짧다(평균 101토큰). 하지만 대화의 56%에 UNK가 있고, **정답에서도 글자가 사라진다.**

In [2]:
ke = next(r for r in rep["tokenizers"] if r["model"].startswith("KETI"))
print("UNK 로 바뀐 글자 top:", ke["source_unk_top"])
print()
for x in ke["target_roundtrip_fail_examples"]:
    print("정답  :", x["orig"])
    print("되살림:", x["decoded"])
    print()

UNK 로 바뀐 글자 top: {'걍': 65, '넹': 61, '앜': 51, '욜': 42, 'ㅏ': 27, 'ㅔ': 27, '겡': 18, '봣': 18, '넼': 17, '믄': 17, 'ㅈ': 15, '왱': 15, 'ㅁ': 15, '댜': 14, 'ㅓ': 14}

정답  : 어제 새벽 4시 30분에 잤지만 7시간을 자 쌩쌩하다고 이야기한다.
되살림: 어제 새벽 4시 30분에 잤지만 7시간을 자 하다고 이야기한다.

정답  : 혼자 만날 때도 그 사람의 성격을 알 수 있지만 여러 명이서 만나면 다른 사람에게 대하는 모습도 알 수 있다.
되살림: 혼자 만날 때도 그 사람의 성격을 알 수 있지만 여러 명이서 만나면 다른 사람에게 대하는 모습도 알 수 있다.

정답  : 사진이 정말 얄밉게 나왔다고 이야기하고 있다.
되살림: 사진이 정말 얄게 나왔다고 이야기하고 있다.

정답  : 러스크 수지에 있는 선생님(쌤)들 중에 몇 명은 좋은데 나머지는 아닌 것 같다.
되살림: 러스크 수지에 있는 선생님()들 중에 몇 명은 좋은데 나머지는 아닌 것 같다.

정답  : 팔걸이에 기대서 졸다가 팔을 떨어트려서 부끄러운 마음에 잠이 확 깼다.
되살림: 팔걸이에 기대서 졸다가 팔을 떨어트려서 부끄러운 마음에 잠이 확 다.



`쌩쌩하다 → 하다`, `얄밉게 → 얄게`, `깼다 → 다`. 이 토크나이저로 학습한 모델은 이 단어들을 절대 출력할 수 없다.

## 2. mt5 — UNK는 적지만 무겁다

UNK 비율은 0.34%지만 대화의 31%에 하나 이상 있다. vocab이 25만이라 입력 임베딩만 약 1.9억 파라미터로, 8GB GPU에서 불리하다.

In [3]:
mt = next(r for r in rep["tokenizers"] if r["model"].startswith("google"))
print("UNK 로 바뀐 글자 top:", mt["source_unk_top"])

UNK 로 바뀐 글자 top: {'앜': 51, '겡': 18, '넼': 18, '왱': 15, '뎈': 13, '먄': 13, '즁': 13, '짘': 13, '왤': 12, '낰': 12, '닼': 11, '얔': 10, '됫': 9, '곸': 9, '묜': 9}


## 3. pko-t5 — UNK 0, 개행 보존

바이트 단위 BPE라 원리상 UNK가 나오지 않는다. 한국어 특수 표기가 어떻게 쪼개지는지 본다.

In [4]:
pko = rep["tokenizers"][0]
pd.DataFrame({m: {p: " | ".join(t) for p, t in r["probes"].items()}
              for m, r in [(r["model"], r) for r in rep["tokenizers"]] if "small" not in m})

,paust/pko-t5-base,KETI-AIR/ke-t5-base,google/mt5-base
ㅋㅋ,ᄏ | ᄏ,ᄏᄏ,| ᄏᄏ
ㅠㅠ,ᅲ | ᅲ,ᅲᅲ,| ᅲᅲ
ㄱㅅ,ᄀ | ᄉ,ᄀ | ᄉ,| ᄀ | ᄉ
#@이름#,# | @ | 이름 | #,# | @ | 이름 | #,# | @ | 이 | 름 | #
#@기타#,# | @ | 기타 | #,# | @ | 기타 | #,# | @ | 기 | 타 | #
[사진],[ | 사진 | ],[ | 사진 | ],[ | 사진 | ]
😂,� | � | � | �,| <unk>,| 😂
♡,� | � | �,♡,| ♡
~~,~ | ~,| ~~,| ~~
먹을래?,먹을래 | ?,먹을 | 래 | ?,먹 | 을 | 래 | ?


- `#@이름#`은 pko-t5에서 `#`, `@`, `이름`, `#` 4토큰이다. special token으로 추가하지 않기로 했다 (근거는 실험 로그)
- `ke-t5`, `mt5`는 개행(`
`)을 공백으로 바꾼다. `speaker` 포맷의 줄 구분이 사라진다

## 4. pko-t5의 정답 되살림이 100%가 아닌 이유 — NFKC

정답 되살림 99.73%에서 실패한 4건은 눈으로는 똑같다. pko-t5 토크나이저가 입력에 **NFKC 유니코드 정규화**를 적용하기 때문이다.

In [5]:
for x in pko["target_roundtrip_fail_examples"][:2]:
    o, d = x["orig"], x["decoded"]
    diff = [(c, f"U+{ord(c):04X}", unicodedata.name(c, "?")) for c in set(o) - set(d)]
    print(o)
    print("  원문에만 있는 글자:", diff, "/ NFKC(원문) == 되살림:", unicodedata.normalize("NFKC", o) == d)

혼자 만날 때도 그 사람의 성격을 알 수 있지만 여러 명이서 만나면 다른 사람에게 대하는 모습도 알 수 있다.
  원문에만 있는 글자: [('\xa0', 'U+00A0', 'NO-BREAK SPACE')] / NFKC(원문) == 되살림: True
집에 누가 있는지, 엄마도 있는지 물어본다.
  원문에만 있는 글자: [('\xa0', 'U+00A0', 'NO-BREAK SPACE')] / NFKC(원문) == 되살림: True


In [6]:
# 호환 자모도 NFKC 에서 조합용 자모로 바뀐다
for c in "ㅋㅎㅠ":
    n = unicodedata.normalize("NFKC", c)
    print(c, f"U+{ord(c):04X}", "->", n, f"U+{ord(n):04X}", unicodedata.name(n))

ㅋ U+314B -> ᄏ U+110F HANGUL CHOSEONG KHIEUKH
ㅎ U+314E -> ᄒ U+1112 HANGUL CHOSEONG HIEUH
ㅠ U+3160 -> ᅲ U+1172 HANGUL JUNGSEONG YU


**결과가 평가에 주는 영향**: 모델 출력은 항상 NFKC 형태로 나온다. 정답에 줄바꿈 없는 공백(U+00A0)이 있으면,
완벽한 출력도 ROUGE가 깎인다. 그래서 평가할 때 **예측과 정답 양쪽에 NFKC를 적용**한다 (`docs/03_evaluation.md`).
정답 파일 자체는 바꾸지 않는다.

## 결정

| 항목 | 선택 | 되돌릴 조건 |
|---|---|---|
| 모델 | `paust/pko-t5-base` (2.76억) | VRAM·시간이 부족하면 `pko-t5-small` (토크나이저가 같아 전처리 그대로) |
| special token | 추가하지 않음 | 출력에 태그 조각이 섞이는 오류가 주요 원인으로 나오면 단일 변수로 비교 |
| 평가 정규화 | 예측·정답 양쪽 NFKC | — |